In [1]:
import math
import numpy as np
import meshcat_shapes
import pink

import numpy.typing as npt
from pink.utils import custom_configuration_vector
from pink.visualization import start_meshcat_visualizer
from robot_descriptions.loaders.pinocchio import load_robot_description
from scipy.spatial.transform import Rotation as R

In [2]:
robot_joint_pos = {
    "top": (
        math.radians(-90.17),
        math.radians(-105.66),
        math.radians(-63),
        math.radians(-101.35),
        math.radians(90),
        math.radians(0),
    ),
}

def pose_to_TMatrix(pose: list[float]|npt.ArrayLike):
    """
    Converts a pose represented by a list or array-like object into a 
    4x4 transformation matrix.
    
    Args:
        pose : A list or array-like object containing six elements representing the 
        pose of the form [x, y, z, rx, ry, rz).
    """

    x, y, z, rx, ry, rz = pose
    rot = R.from_rotvec([rx, ry, rz])
    m = np.eye(4)
    m[:3, :3] = rot.as_matrix()
    m[:3, 3] = [x, y, z]
    return m

def TMatrix_to_pose(m: npt.NDArray):
    """
    Converts a 4x4 transformation matrix to a pose representation.

    Returns:
        A list containing the translation and rotation in the form 
        [tx, ty, tz, rx, ry, rz], where (tx, ty, tz) is the translation 
        vector and (rx, ry, rz) is the rotation vector in radians.
    """

    rot = R.from_matrix(m[:3, :3])
    t = m[:3, 3]
    rx, ry, rz = rot.as_rotvec()
    return [t[0], t[1], t[2], rx, ry, rz]

def inverse_TMatrix(T: npt.NDArray):
    """
    Compute the inverse of a 4x4 transformation matrix.
    """

    R = T[:3, :3]
    t = T[:3, 3]
    R_inv = R.T
    t_inv = -np.dot(R_inv, t)
    T_inv = np.eye(4)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv
    return T_inv

def compare_transforms(T_a, T_b):
    """
    Compares two 4x4 transformation matrices T_a and T_b, and calculates the translation error, rotation error, and the relative transformation between them.
    
    Example usage:
    >>> t_err, r_err, T_rel = compare_transforms(T_x, pose_to_TMatrix(actual_cobot_tcp))
    >>> print("translation error [m]:", np.round(t_err, 3))
    >>> print("rotation error [deg]:", np.round(np.degrees(r_err),3) )
    >>> print("relative transform T_robot^-1 @ T_sim:")
    >>> print(np.array_str(T_rel, precision=3, suppress_small=True))
    """
    T_rel = np.linalg.inv(T_a) @ T_b
    t_err = np.linalg.norm(T_rel[:3, 3])
    r_err = np.linalg.norm(R.from_matrix(T_rel[:3, :3]).as_rotvec())
    # fro_err = np.linalg.norm(T_a - T_b, ord="fro") # not meaninful
    
    print("translation error:", np.round(t_err, 3))
    print("rotation error:", np.round(np.degrees(r_err),3) )
    print("relative transform T_robot^-1 @ T_sim:")
    print(np.array_str(T_rel, precision=3, suppress_small=True))
    return t_err, r_err, T_rel



In [3]:
def move_sim_robot():
    from rtde_control import RTDEControlInterface
    from rtde_receive import RTDEReceiveInterface

    robot_ip = "192.168.174.130"  # set to your robot controller IP

    rtde_c = RTDEControlInterface(robot_ip)
    rtde_r = RTDEReceiveInterface(robot_ip)

    # move robot to inital pose
    rtde_c.moveJ(list(robot_joint_pos["top"]), speed=0.5, acceleration=0.5, asynchronous = False)

    rtde_c.disconnect()
    rtde_r.disconnect()
    
    print("Robot moved to initial pose")

In [4]:
def get_actual_tcp_pose():
    from rtde_receive import RTDEReceiveInterface

    rtde_r = RTDEReceiveInterface("192.168.174.130")
    tcp_pose = rtde_r.getActualTCPPose()
    rtde_r.disconnect()
    return tcp_pose  # [x, y, z, rx, ry, rz] in base frame, rotvec

In [5]:
robot = load_robot_description("ur5_official_description")

In [6]:
viz = start_meshcat_visualizer(robot)
viewer = viz.viewer
meshcat_shapes.frame(viewer["end_effector_target"], opacity=0.5)
meshcat_shapes.frame(viewer["end_effector"], opacity=1.0)

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7000/static/


In [7]:
joint_names = [
    "shoulder_pan_joint",
    "shoulder_lift_joint",
    "elbow_joint",
    "wrist_1_joint",
    "wrist_2_joint",
    "wrist_3_joint",
]

# set initial configuration for visualization
q_kwargs = dict(zip(joint_names, robot_joint_pos["top"]))

q_ref = custom_configuration_vector(
    robot, **q_kwargs
)

configuration = pink.Configuration(robot.model, robot.data, q_ref)
viz.display(configuration.q) # q is the same a s q_ref

In [8]:
move_sim_robot()

Robot moved to initial pose


## actual robot

In [58]:
#  # tool relative to base
actual_cobot_tcp =  get_actual_tcp_pose()
T_actual_cobot_tcp = pose_to_TMatrix(actual_cobot_tcp)

print(f"actual_cobot_tcp\n{actual_cobot_tcp}")
print()
print(np.array_str(T_actual_cobot_tcp, precision=3, suppress_small=True))


actual_cobot_tcp
[-0.1350765176681322, -0.5985473191433881, 0.5492253402726713, 0.00466064408878731, 3.141588925394616, 0.0002741556748556326]

[[-1.     0.003  0.    -0.135]
 [ 0.003  1.     0.    -0.599]
 [-0.     0.    -1.     0.549]
 [ 0.     0.     0.     1.   ]]


# pink library

In [ ]:
# get rel to world
X_w2tool = configuration.get_transform_frame_to_world("tool0")
X_w2base = configuration.get_transform_frame_to_world("base")

print(f"X_w2tool\n{X_w2tool}")
print(f"X_w2base\n{X_w2base}")

In [ ]:
# T_base2tool

# high level API
T_x_highlevelapi = configuration.get_transform("tool0", "base")
print(f">>T_base2tool high level api:\n{T_x_highlevelapi}")

# more manual way
X_base2tool = X_w2base.inverse() * X_w2tool
T_x = np.eye(4)
T_x[:3, :3] = X_base2tool.rotation
T_x[:3, 3] = X_base2tool.translation
print(f">>T_base2tool manual:\n{np.array_str(T_x, precision=6, suppress_small=True)}\n")

print(">>diff between high level api and manual")
_ = compare_transforms(T_x_highlevelapi, T_x)

print("\n>>diff between manual and actual")
_ = compare_transforms(pose_to_TMatrix(actual_cobot_tcp), T_x)

>>T_base2tool high level api:
  R =
   -0.999996   0.00296706  5.18055e-07
  0.00296706     0.999996  0.000174532
-2.05139e-10  0.000174533           -1
  p = -0.110912 -0.593621  0.493195

>>T_base2tool manual:
[[-0.999996  0.002967  0.000001 -0.110912]
 [ 0.002967  0.999996  0.000175 -0.593621]
 [-0.        0.000175 -1.        0.493195]
 [ 0.        0.        0.        1.      ]]

>>diff between high level api and manual
translation error: 0.0
rotation error: 0.0
relative transform T_robot^-1 @ T_sim:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

>>diff between manual and actual
translation error: 0.061
rotation error: 0.0
relative transform T_robot^-1 @ T_sim:
[[ 1.    -0.     0.    -0.024]
 [ 0.     1.    -0.     0.005]
 [-0.     0.     1.     0.056]
 [ 0.     0.     0.     1.   ]]

>>diff between high level and actual
translation error: 0.061
rotation error: 0.0
relative transform T_robot^-1 @ T_sim:
[[ 1.     0.    -0.     0.024]
 [-0.     1.     0.    -0.005]
 [ 